In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import glob
import subprocess

In [ ]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    whl_files = sorted(glob.glob(f'{temp_dir}/wheels/*.whl'))
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-deps', 
        *whl_files
    ], check=True)

In [ ]:
set_env(
    input_archive='/kaggle/input/notebooks/nahidhossainredom/aimo-utils-qwen-3-5/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'

In [ ]:
import gc
import re
import json
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [ ]:
class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )
    
    tool_description = (
        'Execute Python code in a stateful Jupyter notebook environment.\n'
        'Use this for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Brute-force verification for small cases\n\n'
        'Available libraries: math, numpy, sympy, itertools, collections, mpmath.\n'
        'Always use print() to display results. Code persists between executions.'
    )
    
    preference_prompt = (
        'You have access to a Python tool for computation. Use it when helpful.\n'
        'Available libraries: `math`, `numpy`, `sympy`, `itertools`, `collections`, `mpmath`.\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Validate computational results against known cases or theoretical bounds'
    )
    
    served_model_name = 'nemotron'
    model_path = '/kaggle/input/models/barnobarno/nvidia-nemotron-3-super-120b-a12b-nvfp4/transformers/nvfp4/1'
    reasoning_parser_path = ''
    
    kv_cache_dtype = 'fp8'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    context_tokens = 65536
    top_logprobs = 5
    batch_size = 512
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.9
    temperature = 1.0
    top_p = 0.95
    top_k = 20
    min_p = 0.02

    # Local validation output directories (not used during competition rerun)
    reasoning_dir = 'reasoning_traces'


In [ ]:
set_seed(CFG.seed)

In [ ]:
PYTHON_TOOL_DEFINITION = {
    'type': 'function',
    'function': {
        'name': 'python',
        'description': CFG.tool_description,
        'parameters': {
            'type': 'object',
            'properties': {
                'code': {
                    'type': 'string',
                    'description': 'Python code to execute'
                }
            },
            'required': ['code']
        }
    }
}

In [ ]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [ ]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    def execute(self, code: str) -> str:

        self._ensure_session()
        final_script = self._ensure_last_print(code)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return output

In [ ]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'

        self.problem_counter = 0

        # Create output dirs for local validation runs
        if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            os.makedirs(self.cfg.reasoning_dir, exist_ok=True)
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
    
    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
    
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--pipeline-parallel-size',
            '1',
            '--data-parallel-size',
            '1',
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--trust-remote-code', 
            '--swap-space', 
            '0', 
            '--enable-chunked-prefill', 
            '--enable-auto-tool-choice', 
            '--attention-backend FLASHINFER',
            '--tool-call-parser', 
            'qwen3_coder', 
            '--reasoning-parser-plugin',
            self.cfg.reasoning_parser_path, 
            '--reasoning-parser', 
            'super_v3',
            '--async-scheduling', 
            '--disable-log-stats'
        ]
    
        self.log_file = open('vllm_server.log', 'w')
    
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
    
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> int | None:
        
        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
                
        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
    
        return None
    
    def _compute_mean_entropy(self, logprobs_list: list) -> float:
    
        if not logprobs_list:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for token_logprob_info in logprobs_list:
            
            if not hasattr(token_logprob_info, 'top_logprobs') or not token_logprob_info.top_logprobs:
                continue
            
            token_entropy = 0.0
            
            for top_lp in token_logprob_info.top_logprobs:
                prob = math.exp(top_lp.logprob)
                
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        if token_count == 0:
            return float('inf')
    
        return total_entropy / token_count

    def _extract_tool_code(self, tool_calls) -> str | None:

        if not tool_calls:
            return None
    
        for tc in tool_calls:
            if tc.function.name == 'python':
                try:
                    args = json.loads(tc.function.arguments)
                    return args.get('code', '')
                except (json.JSONDecodeError, AttributeError):
                    return tc.function.arguments
    
        return None

    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:
    
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1,
                'Answer': None,
                'Turns': 0,
                'Response Length': 0,
                'Thinking Tokens': 0,
                'Gen Time': 0.0,
                'Tokens/sec': 0.0,
                'Python Calls': 0,
                'Python Errors': 0,
                'Entropy': float('inf'),
                'FullReasoning': '[SKIPPED]'
            }
    
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        thinking_tokens = 0
        final_answer = None
        turn_number = 0
        gen_time = 0.0

        logprobs_buffer = []
        reasoning_parts = []

        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
    
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout,
                sandbox=sandbox
            )
    
            messages = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': problem}
            ]
    
            for turn_idx in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    reasoning_parts.append('\n--- [STOPPED: early stop or deadline] ---\n')
                    break

                turn_number += 1
                turn_start = time.time()

                try:
                    response = self.client.chat.completions.create(
                        model=self.cfg.served_model_name,
                        messages=messages,
                        tools=[PYTHON_TOOL_DEFINITION],
                        temperature=self.cfg.temperature,
                        top_p=self.cfg.top_p,
                        seed=attempt_seed,
                        logprobs=True,
                        top_logprobs=self.cfg.top_logprobs,
                        extra_body={
                            'top_k': self.cfg.top_k,
                            'min_p': self.cfg.min_p
                        }
                    )
                except Exception as api_exc:
                    reasoning_parts.append(
                        f'\n--- [API ERROR turn {turn_number}: {type(api_exc).__name__}: {api_exc}] ---\n'
                    )
                    print(f'[Attempt {attempt_index+1}] API error: {type(api_exc).__name__}: {api_exc}')
                    break

                turn_elapsed = time.time() - turn_start
                gen_time += turn_elapsed

                choice = response.choices[0]
                assistant_message = choice.message
                finish_reason = choice.finish_reason

                turn_completion_tokens = (response.usage.completion_tokens or 0) if response.usage else 0
                total_tokens += turn_completion_tokens

                # Extract thinking/reasoning content populated by vLLM reasoning parser
                reasoning_content = getattr(assistant_message, 'reasoning_content', None)
                turn_thinking_words = 0

                if reasoning_content:
                    turn_thinking_words = len(reasoning_content.split())
                    thinking_tokens += turn_thinking_words
                    reasoning_parts.append(
                        f'[Turn {turn_number} | Thinking ({turn_thinking_words} words)]\n'
                        f'{reasoning_content}\n'
                    )

                if choice.logprobs and choice.logprobs.content:
                    logprobs_buffer.extend(choice.logprobs.content)

                content_text = assistant_message.content or ''
                turn_tps = turn_completion_tokens / turn_elapsed if turn_elapsed > 0 else 0.0

                reasoning_parts.append(
                    f'[Turn {turn_number} | finish={finish_reason} | '
                    f'{turn_completion_tokens} tok | {turn_tps:.1f} tok/s]\n'
                    f'{content_text}\n'
                )

                answer = self._scan_for_answer(content_text)
                if answer is not None:
                    final_answer = answer
                    reasoning_parts.append(f'[Found answer: {answer}]\n')
                    break

                tool_calls = assistant_message.tool_calls
                code = self._extract_tool_code(tool_calls)

                if code is not None:
                    python_calls += 1
                    reasoning_parts.append(
                        f'[Turn {turn_number} | Tool Call]\n```python\n{code}\n```\n'
                    )

                    messages.append({
                        'role': 'assistant',
                        'content': content_text if content_text else None,
                        'tool_calls': [
                            {
                                'id': tc.id,
                                'type': 'function',
                                'function': {
                                    'name': tc.function.name,
                                    'arguments': tc.function.arguments
                                }
                            }
                            for tc in tool_calls
                        ]
                    })

                    output = local_tool.execute(code)
                    reasoning_parts.append(f'[Turn {turn_number} | Tool Output]\n{output}\n')

                    if output.startswith('[ERROR]') or 'Traceback' in output or 'Error:' in output:
                        python_errors += 1

                    messages.append({
                        'role': 'tool',
                        'tool_call_id': tool_calls[0].id,
                        'content': output
                    })

                else:
                    if finish_reason == 'stop':
                        final_answer = self._scan_for_answer(content_text)
                        if final_answer is not None:
                            reasoning_parts.append(f'[Found answer: {final_answer}]\n')
                    reasoning_parts.append(
                        f'\n--- [STOPPED: finish_reason={finish_reason}, no tool calls] ---\n'
                    )
                    break
    
        except Exception as exc:
            python_errors += 1
            reasoning_parts.append(f'\n--- [EXCEPTION: {exc}] ---\n')
    
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
    
        mean_entropy = self._compute_mean_entropy(logprobs_buffer)
        tps = total_tokens / gen_time if gen_time > 0 else 0.0

        return {
            'Attempt': attempt_index + 1,
            'Answer': final_answer,
            'Turns': turn_number,
            'Response Length': total_tokens,
            'Thinking Tokens': thinking_tokens,
            'Gen Time': round(gen_time, 2),
            'Tokens/sec': round(tps, 1),
            'Python Calls': python_calls,
            'Python Errors': python_errors,
            'Entropy': mean_entropy,
            'FullReasoning': '\n'.join(reasoning_parts)
        }

    def _save_reasoning_csv(
        self,
        detailed_results: list,
        problem_id: int,
        problem_text: str,
        ground_truth: int | None = None
    ) -> str | None:

        if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            return None

        rows = []
        for r in sorted(detailed_results, key=lambda x: x['Attempt']):
            answer = r['Answer']
            if answer is None:
                verdict = 'unfinished'
            elif ground_truth is not None:
                verdict = 'correct' if answer == ground_truth else 'wrong'
            else:
                verdict = 'unknown'

            rows.append({
                'problem_id': problem_id,
                'problem_text': problem_text[:500],
                'attempt': r['Attempt'],
                'answer': answer if answer is not None else '',
                'ground_truth': ground_truth if ground_truth is not None else '',
                'verdict': verdict,
                'turns': r.get('Turns', 0),
                'total_tokens': r['Response Length'],
                'thinking_tokens': r.get('Thinking Tokens', 0),
                'gen_time': r.get('Gen Time', 0.0),
                'tokens_per_sec': r.get('Tokens/sec', 0.0),
                'python_calls': r['Python Calls'],
                'python_errors': r['Python Errors'],
                'full_reasoning': r.get('FullReasoning', '')
            })

        if not rows:
            return None

        df = pd.DataFrame(rows)
        os.makedirs(self.cfg.reasoning_dir, exist_ok=True)
        csv_path = os.path.join(self.cfg.reasoning_dir, f'problem_{problem_id}_reasoning.csv')
        df.to_csv(csv_path, index=False)

        verdicts = df['verdict'].value_counts().to_dict()
        verdict_str = ', '.join(f'{v}: {c}' for v, c in verdicts.items())
        print(f'Reasoning saved: {csv_path} ({len(rows)} attempts | {verdict_str})')

        return csv_path

    def _select_answer(self, detailed_results: list) -> int:

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            
            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)
                
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer, 
                'votes': answer_votes[answer], 
                'score': total_weight
            })

        scored_answers.sort(key=lambda x: x['score'], reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'], 
                item['votes'], 
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data, 
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)
        
        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer
    
    def solve_problem(self, problem: str, ground_truth_answer: int | None = None) -> int:

        problem_start_time = time.time()
        self.problem_counter += 1
        problem_id = self.problem_counter
    
        print(f'\nProblem {problem_id}: {problem[:200]}...\n')
        
        user_input = f'{problem}\n\n{self.cfg.preference_prompt}'
    
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
    
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
    
        deadline = time.time() + budget
    
        print(f'Budget: {budget:.2f}s | Problems remaining: {self.problems_remaining}\n')
    
        tasks = []
    
        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))
    
        detailed_results = []
        valid_answers = []
    
        stop_event = threading.Event()
    
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
    
        try:
            futures = []
    
            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline
                )
    
                futures.append(future)
    
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
    
                    counts = Counter(valid_answers).most_common(1)
    
                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()
    
                        for f in futures:
                            f.cancel()
    
                        break
    
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
    
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            
            self.problems_remaining = max(0, self.problems_remaining - 1)

        used_time = time.time() - problem_start_time
    
        if detailed_results:
            results_dataframe = pd.DataFrame([
                {
                    'Attempt': r['Attempt'],
                    'Answer': r['Answer'],
                    'Turns': r.get('Turns', 0),
                    'Tokens': r['Response Length'],
                    'Thinking': r.get('Thinking Tokens', 0),
                    'Tok/s': r.get('Tokens/sec', 0.0),
                    'Gen Time': r.get('Gen Time', 0.0),
                    'Py Calls': r['Python Calls'],
                    'Py Errs': r['Python Errors'],
                    'Entropy': round(r['Entropy'], 3) if r['Entropy'] != float('inf') else None,
                }
                for r in detailed_results
            ])
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            display(results_dataframe)

            total_tokens_all = sum(r['Response Length'] for r in detailed_results)
            total_gen_time_all = sum(r.get('Gen Time', 0.0) for r in detailed_results)
            overall_tps = total_tokens_all / total_gen_time_all if total_gen_time_all > 0 else 0
            print(
                f'\n[Timing] Wall time: {used_time:.1f}s | '
                f'Total tokens: {total_tokens_all} | '
                f'Overall throughput: {overall_tps:.1f} tok/s\n'
            )
    
        if not valid_answers:
            print('\nResult: 0\n')
            self._save_reasoning_csv(detailed_results, problem_id, problem, ground_truth_answer)
            return 0

        final_answer = self._select_answer(detailed_results)
        self._save_reasoning_csv(detailed_results, problem_id, problem, ground_truth_answer)

        return final_answer
    
    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass


In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
# ── Server Diagnostics ─────────────────────────────────────────────────────
# Verify models are available and check server logs for errors/warnings.
# Key things to look for:
#   - "reasoning parser" lines → confirms nemotron_v3 parser loaded OK
#   - "tool call parser" lines → confirms qwen3_coder parser loaded OK
#   - Any ERROR/WARNING messages at startup

models = solver.client.models.list()
print(f'Available models: {[m.id for m in models.data]}\n')

with open('vllm_server.log') as _f:
    _lines = _f.readlines()

print(f'vllm_server.log ({len(_lines)} lines total) — last 60 lines:')
print('─' * 80)
print(''.join(_lines[-60:]))
print('─' * 80)


In [ ]:
# ── Tool Call & Throughput Test ────────────────────────────────────────────
# Sends a trivial math problem that should trigger a Python tool call.
# Reports:
#   finish_reason  → should be 'tool_calls' if tool calling works
#   reasoning_content → should be non-None if --reasoning-parser is active
#   tok/s          → baseline single-turn throughput

_test_messages = [
    {'role': 'system', 'content': 'You are a math assistant. Use the python tool to compute.'},
    {'role': 'user',   'content': 'What is 1234 * 5678? Use the python tool to compute it.'},
]

print('Sending test request (tool call + throughput check)...\n')
_t0 = time.time()

_resp = solver.client.chat.completions.create(
    model=CFG.served_model_name,
    messages=_test_messages,
    tools=[PYTHON_TOOL_DEFINITION],
    temperature=0.0,
    max_tokens=1024,
    seed=42,
)

_elapsed = time.time() - _t0
_choice = _resp.choices[0]
_ctok = _resp.usage.completion_tokens if _resp.usage else 0
_tps = _ctok / _elapsed if _elapsed > 0 else 0.0

print(f'finish_reason      : {_choice.finish_reason}')
print(f'completion_tokens  : {_ctok}')
print(f'time               : {_elapsed:.2f}s')
print(f'throughput         : {_tps:.1f} tok/s')
print()

# Tool call info
if _choice.message.tool_calls:
    for _tc in _choice.message.tool_calls:
        print(f'tool_call.name : {_tc.function.name}')
        print(f'tool_call.args : {_tc.function.arguments[:300]}')
else:
    print('⚠  No tool_calls in response (finish_reason should have been tool_calls)')
    print(f'content: {repr((_choice.message.content or "")[:500])}')

# Reasoning / thinking content (populated by --reasoning-parser)
_rc = getattr(_choice.message, 'reasoning_content', None)
print(f'\nreasoning_content present: {_rc is not None}')
if _rc:
    print(f'reasoning_content (first 400 chars):\n{_rc[:400]}')
else:
    print('  → Either parser is not active or thinking tokens are empty for this request.')


In [ ]:
# ── Validation Data Loader ─────────────────────────────────────────────────
# Loads a dataset with ground-truth answers for local accuracy tracking.
# Tries multiple known paths; falls back to competition reference CSV.
# Set VALIDATION_CSV to override.

VALIDATION_CSV = '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv'

_candidate_paths = [
    VALIDATION_CSV,
    '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv',
    '/kaggle/input/datasets/nahidhossainredom/omni-math-hardestdifficulty-9/omni_math_hard.csv',
    '../OlymMATH-EN-HARD.csv',
]

_val_df = None
for _p in _candidate_paths:
    if _p and os.path.exists(_p):
        _val_df = pd.read_csv(_p)
        print(f'Loaded validation data from: {_p}')
        break

if _val_df is None:
    print('No validation CSV found — ground truth will be unavailable. '
          'Set VALIDATION_CSV at the top of this cell to a local path.')
    ground_truth = {}
    _val_df = pd.DataFrame(columns=['id', 'problem'])
else:
    ground_truth = (
        dict(zip(_val_df['id'], _val_df['answer']))
        if 'answer' in _val_df.columns else {}
    )
    print(f'Problems: {len(_val_df)} | Ground truth available: {len(ground_truth)}')

_val_df.drop('answer', axis=1, errors='ignore').to_csv('reference.csv', index=False)

predictions = {}
correct_count = 0
total_count = 0


In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions
    
    id_value = id_.item(0)
    question_text = question.item(0)

    print('------')
    print(f'ID: {id_value}')
    print(f'Question: {question_text[:200]}...')

    gt_answer = ground_truth.get(id_value, None)

    gc.disable()

    final_answer = solver.solve_problem(question_text, ground_truth_answer=gt_answer)
    predictions[id_value] = final_answer

    gc.enable()
    gc.collect()

    total_count += 1
    if id_value in ground_truth:
        gt = ground_truth[id_value]
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = 'CORRECT' if is_correct else 'WRONG'
        acc = 100 * correct_count / total_count
        print(f'Answer: {final_answer} | Ground Truth: {gt} | {status}')
        print(f'Running Accuracy: {correct_count}/{total_count} ({acc:.1f}%)')
    else:
        print(f'Answer: {final_answer}')

    print('------\n')

    return pl.DataFrame({'id': id_value, 'answer': final_answer})


In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('reference.csv',))
